In [1]:
import sys
!{sys.executable} -m pip install deap numpy


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: /Users/vikasshukla/.pyenv/versions/3.11.6/bin/python -m pip install --upgrade pip


In [2]:
import operator
import math
import random
import numpy as np

from deap import algorithms, base, creator, tools, gp

In [3]:
def protected_div(left, right):
    try:
        return left / right
    except ZeroDivisionError:
        return 1

In [4]:
pset = gp.PrimitiveSet("MAIN", 1)

pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(protected_div, 2)
pset.addPrimitive(operator.neg, 1)
pset.addPrimitive(math.cos, 1)
pset.addPrimitive(math.sin, 1)

pset.addEphemeralConstant("rand101", lambda: random.randint(-1, 1))

pset.renameArguments(ARG0="x")

/Users/vikasshukla/.pyenv/versions/3.11.6/lib/python3.11/site-packages/deap/gp.py:257: RuntimeWarning: Ephemeral rand101 function cannot be pickled because its generating function is a lambda function. Use functools.partial instead.
  warnings.warn("Ephemeral {name} function cannot be "


In [5]:
if "FitnessMin" not in creator.__dict__:
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

if "IndividualGP" not in creator.__dict__:
    creator.create("IndividualGP", gp.PrimitiveTree, fitness=creator.FitnessMin)

In [6]:
toolbox = base.Toolbox()

toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=2)
toolbox.register("individual", tools.initIterate, creator.IndividualGP, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("compile", gp.compile, pset=pset)

In [7]:
def eval_func(individual):
    func = toolbox.compile(expr=individual)

    points = [x / 10.0 for x in range(-10, 10)]

    sqerrors = (
        (func(x) - (x**4 + x**3 + x**2 + x)) ** 2
        for x in points
    )

    return math.fsum(sqerrors) / len(points),

In [8]:
toolbox.register("evaluate", eval_func)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", gp.cxOnePoint)

toolbox.register("expr_mut", gp.genFull, min_=0, max_=2)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr_mut, pset=pset)

In [9]:
toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=17))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=17))

In [10]:
random.seed(7)

population = toolbox.population(n=500)
hall_of_fame = tools.HallOfFame(1)

In [11]:
stats_fit = tools.Statistics(lambda individual: individual.fitness.values)
stats_size = tools.Statistics(len)

stats = tools.MultiStatistics(fitness=stats_fit, size=stats_size)

stats_fit.register("avg", np.mean)
stats_fit.register("std", np.std)
stats_fit.register("min", np.min)
stats_fit.register("max", np.max)

stats_size.register("avg", np.mean)
stats_size.register("std", np.std)
stats_size.register("min", np.min)
stats_size.register("max", np.max)

In [12]:
population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=0.5,
    mutpb=0.1,
    ngen=40,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

   	      	                        fitness                        	                      size                     
   	      	-------------------------------------------------------	-----------------------------------------------
gen	nevals	avg    	gen	max    	min     	nevals	std    	avg  	gen	max	min	nevals	std    
0  	500   	2.25895	0  	20.7006	0.165572	500   	3.33602	3.738	0  	7  	2  	500   	1.62522
1  	261   	1.01805	1  	6.85956	0.165572	261   	0.646451	3.646	1  	10 	1  	261   	1.69608
2  	280   	1.00755	2  	20.7006	0.165572	280   	1.51513 	3.962	2  	12 	1  	280   	1.87631
3  	287   	0.889995	3  	16.2332	0.165572	287   	0.965466	4.294	3  	12 	1  	287   	2.13063
4  	304   	0.827531	4  	16.3438	0.101561	304   	0.892789	4.478	4  	14 	1  	304   	2.26219
5  	273   	0.755672	5  	16.5546	0.165572	273   	0.927691	4.958	5  	17 	1  	273   	2.33843
6  	275   	0.589518	6  	4.09456	0.101561	275   	0.544382	5.534	6  	14 	1  	275   	2.21017
7  	273   	0.48352 	7  	6.3679 	0.101561	273   	0.556064

In [13]:
best_individual = hall_of_fame[0]

print("Best individual:")
print(best_individual)

print("\nBest fitness:")
print(best_individual.fitness.values[0])

Best individual:
add(mul(x, x), add(mul(mul(add(mul(x, x), x), x), x), x))

Best fitness:
4.198527278764174e-33


In [14]:
func = toolbox.compile(expr=best_individual)

print("x | predicted | actual")
print("----------------------")

for x in [-1, -0.5, 0, 0.5, 1]:
    predicted = func(x)
    actual = x**4 + x**3 + x**2 + x

    print(x, "|", round(predicted, 4), "|", round(actual, 4))

x | predicted | actual
----------------------
-1 | 0 | 0
-0.5 | -0.3125 | -0.3125
0 | 0 | 0
0.5 | 0.9375 | 0.9375
1 | 4 | 4
